# 1D CNN
Input: the power spectrum (1-45 Hz) of all 19 channels for one epoch. The CNN slides along frequency. Same subject-level CV as the baselines.

In [1]:
import sys
sys.path.insert(0, "..")

import pandas as pd
from src.features import load_spectra
from src.models import make_cnn, spectra_table
from src.evaluate import repeat_cv
from src.config import RESULTS

spectra = spectra_table(load_spectra())          # one row per epoch: 19 channels x 89 frequencies
print(spectra.shape)

rows = []
for classes in [["AD", "CN"], ["AD", "FTD", "CN"]]:
    row = repeat_cv(spectra, make_cnn, classes, n_seeds=3)   # 3 shuffles: the CNN is slower to train
    row.update({"task": "-".join(classes), "model": "CNN"})
    rows.append(row)
cnn = pd.DataFrame(rows).round(3)

# next to the baselines from notebook 02
table = pd.concat([pd.read_csv(RESULTS / "baseline_results.csv"), cnn], ignore_index=True)
table = table[["task", "model", "accuracy", "macro_f1", "recall_AD", "recall_CN", "recall_FTD"]]
print(table.to_string(index=False))
table.to_csv(RESULTS / "model_comparison.csv", index=False)

(17419, 1693)


     task        model  accuracy  macro_f1  recall_AD  recall_CN  recall_FTD
    AD-CN       LogReg     0.794     0.792      0.794      0.793         NaN
    AD-CN RandomForest     0.822     0.820      0.808      0.838         NaN
AD-FTD-CN       LogReg     0.565     0.538      0.581      0.766       0.287
AD-FTD-CN RandomForest     0.644     0.588      0.775      0.821       0.217
    AD-CN          CNN     0.677     0.674      0.685      0.667         NaN
AD-FTD-CN          CNN     0.508     0.483      0.537      0.655       0.275
